# Data Quality Profiling — FlowMetrics

**Purpose:** finding duplicates, nulls, broken relationships, outliers values, and formatting inconsistencies.

---

## Checks performed in this notebook

1. Connection & row counts
2. Primary key integrity (duplicate IDs)
3. Foreign key integrity (orphan records)
4. Null analysis
5. Value range and impossibility checks
6. Format and consistency issues
7. Logical / business consistency

## Setup

Connect to MySQL and import the libraries.

In [5]:
import pandas as pd
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# Display settings — show wider tables, more rows
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

# Connection — adjust credentials as needed
DB_USER     = "root"
DB_PASSWORD = "Mmam5100?"
DB_HOST     = "localhost"
DB_PORT     = 3306
DB_NAME     = "flowmetrics"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def q(sql):
    """Shorthand: run a SQL query, return a DataFrame."""
    return pd.read_sql(sql, engine)

# Test the connection
q("SELECT 1 AS connected")


,connected
0,1


---
## 1 · Row Counts vs. Expected

First, confirm every table loaded fully. If counts are off, no point doing deeper checks until that's resolved.

| Table | Expected |
|---|---|
| customers | 800 |
| users | ~9,314 |
| subscriptions | ~1,105 |
| feature_usage | 367,350 |
| support_tickets | ~8,765 |
| payments | ~2,570 |


In [6]:
row_counts = q("""
SELECT 'customers'       AS table_name, COUNT(*) AS row_count FROM customers
UNION ALL SELECT 'users',          COUNT(*) FROM users
UNION ALL SELECT 'subscriptions',  COUNT(*) FROM subscriptions
UNION ALL SELECT 'feature_usage',  COUNT(*) FROM feature_usage
UNION ALL SELECT 'support_tickets',COUNT(*) FROM support_tickets
UNION ALL SELECT 'payments',       COUNT(*) FROM payments
""")
row_counts

,table_name,row_count
0,customers,800
1,users,9314
2,subscriptions,1105
3,feature_usage,367350
4,support_tickets,8765
5,payments,2570


**✏️ Decision:**

> *All tables are loaded successfully*

---
## 2 · Primary Key Integrity

Each table should have unique values in its ID column. Duplicates here would cause double-counting in every downstream aggregation.

### 2.1 · Duplicate customer IDs

In [7]:
q("""
SELECT customer_id, COUNT(*) AS occurrences
FROM customers
GROUP BY customer_id
HAVING COUNT(*) > 1
""")

,customer_id,occurrences


### 2.2 · Duplicate user IDs

In [8]:
q("""
SELECT user_id, COUNT(*) AS occurrences
FROM users
GROUP BY user_id
HAVING COUNT(*) > 1
""")

,user_id,occurrences


**Note:** If this returns 0 rows but the data dictionary says ~8 duplicates exist, the duplicates may be in the *content* (same person, different user_id) rather than the *key*. Check the next cell.

In [9]:
# Check for content-level user duplicates (same email + same customer_id)
q("""
SELECT
    customer_id,
    LOWER(TRIM(email)) AS clean_email,
    COUNT(*) AS occurrences
FROM users
GROUP BY customer_id, LOWER(TRIM(email))
HAVING COUNT(*) > 1
LIMIT 20
""")

,customer_id,clean_email,occurrences
0,CUST_00041,shawn.johnson@rodriguez-mendoza.com,2
1,CUST_00170,shirley.stanley@wolfe.com,2
2,CUST_00285,alex.armstrong@ford.com,2
3,CUST_00440,antonio.nelson@hinton-davis.com,2
4,CUST_00449,sabrina.walsh@pearson-green.com,2
5,CUST_00455,scott.rodriguez@peterson.org,2
6,CUST_00578,mitchell.richard@cox-ellis.com,2
7,CUST_00722,alyssa.tate@hunt.biz,2


### 2.3 · Duplicate subscription IDs

In [10]:
q("""
SELECT subscription_id, COUNT(*) AS occurrences
FROM subscriptions
GROUP BY subscription_id
HAVING COUNT(*) > 1
""")

,subscription_id,occurrences


### 2.4 · Duplicate payment IDs

In [11]:
q("""
SELECT payment_id, COUNT(*) AS occurrences
FROM payments
GROUP BY payment_id
HAVING COUNT(*) > 1
""")

,payment_id,occurrences


### 2.5 · Duplicate ticket IDs

In [12]:
q("""
SELECT ticket_id, COUNT(*) AS occurrences
FROM support_tickets
GROUP BY ticket_id
HAVING COUNT(*) > 1
""")

,ticket_id,occurrences


---
## 3 · Foreign Key Integrity

Every child record should point to an existing parent. If a `users` row references a `customer_id` that doesn't exist, we have an orphan — which usually indicates a data load problem.

### 3.1 · Users with no matching customer

In [13]:
q("""
SELECT COUNT(*) AS orphan_users
FROM users u
LEFT JOIN customers c ON c.customer_id = u.customer_id
WHERE c.customer_id IS NULL
""")

,orphan_users
0,0


### 3.2 · Subscriptions with no matching customer

In [14]:
q("""
SELECT COUNT(*) AS orphan_subscriptions
FROM subscriptions s
LEFT JOIN customers c ON c.customer_id = s.customer_id
WHERE c.customer_id IS NULL
""")

,orphan_subscriptions
0,0


### 3.3 · Feature usage events with no matching user

In [15]:
q("""
SELECT COUNT(*) AS orphan_events
FROM feature_usage f
LEFT JOIN users u ON u.user_id = f.user_id
WHERE u.user_id IS NULL
""")

,orphan_events
0,0


### 3.4 · Tickets with no matching customer

In [16]:
q("""
SELECT COUNT(*) AS orphan_tickets
FROM support_tickets t
LEFT JOIN customers c ON c.customer_id = t.customer_id
WHERE c.customer_id IS NULL
""")

,orphan_tickets
0,0


### 3.5 · Payments with no matching subscription

In [17]:
q("""
SELECT COUNT(*) AS orphan_payments
FROM payments p
LEFT JOIN subscriptions s ON s.subscription_id = p.subscription_id
WHERE s.subscription_id IS NULL
""")

,orphan_payments
0,0


---
## 4 · Null Analysis

For each column, what fraction is null? Some nulls are expected (CSAT scores when customer didn't respond, end_date when subscription is still active). Others would be data quality issues.

### 4.1 · Customers — null counts by column

In [18]:
q("""
SELECT
    SUM(CASE WHEN customer_id         IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN company_name        IS NULL THEN 1 ELSE 0 END) AS null_company_name,
    SUM(CASE WHEN industry            IS NULL THEN 1 ELSE 0 END) AS null_industry,
    SUM(CASE WHEN country             IS NULL THEN 1 ELSE 0 END) AS null_country,
    SUM(CASE WHEN company_size        IS NULL THEN 1 ELSE 0 END) AS null_company_size,
    SUM(CASE WHEN signup_date         IS NULL THEN 1 ELSE 0 END) AS null_signup_date,
    SUM(CASE WHEN acquisition_channel IS NULL THEN 1 ELSE 0 END) AS null_acquisition_channel,
    COUNT(*)                                                     AS total_rows
FROM customers
""")

,null_customer_id,null_company_name,null_industry,null_country,null_company_size,null_signup_date,null_acquisition_channel,total_rows
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,800


### 4.2 · Subscriptions — null counts by column

In [19]:
q("""
SELECT
    SUM(CASE WHEN end_date IS NULL THEN 1 ELSE 0 END) AS null_end_date,
    SUM(CASE WHEN status   IS NULL THEN 1 ELSE 0 END) AS null_status,
    SUM(CASE WHEN mrr      IS NULL THEN 1 ELSE 0 END) AS null_mrr,
    COUNT(*) AS total_rows
FROM subscriptions
""")

,null_end_date,null_status,null_mrr,total_rows
0,180.0,0.0,0.0,1105


**Note on `end_date` nulls:** these are *expected* — they represent currently-active subscriptions.

In [20]:
q("""
SELECT
    CASE WHEN end_date IS NULL THEN 'null end_date' ELSE 'has end_date' END AS end_date_status,
    status,
    COUNT(*) AS rows_
FROM subscriptions
GROUP BY end_date_status, status
ORDER BY end_date_status, status;
""")

,end_date_status,status,rows_
0,has end_date,churned,66
1,has end_date,downgraded,23
2,has end_date,ended,800
3,has end_date,upgraded,36
4,null end_date,active,180


### 4.3 · Support tickets — null counts (especially CSAT)

In [21]:
q("""
SELECT
    SUM(CASE WHEN csat_score       IS NULL THEN 1 ELSE 0 END) AS null_csat,
    SUM(CASE WHEN resolved_date    IS NULL THEN 1 ELSE 0 END) AS null_resolved,
    SUM(CASE WHEN resolution_hours IS NULL THEN 1 ELSE 0 END) AS null_resolution_hours,
    SUM(CASE WHEN category         IS NULL THEN 1 ELSE 0 END) AS null_category,
    COUNT(*) AS total_rows,
    ROUND(100.0 * SUM(CASE WHEN csat_score IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_null_csat
FROM support_tickets
""")

,null_csat,null_resolved,null_resolution_hours,null_category,total_rows,pct_null_csat
0,3063.0,20.0,20.0,0.0,8765,34.9


**✏️ Decision on CSAT nulls:**

> *CSAT being null for ~35% of tickets is expected — these are customers who didn't respond to the post-resolution survey. Treat these as "no response" (informative null), not as missing data to impute. When averaging CSAT, nulls will be excluded.*

---
## 5 · Value Range & Impossibility Checks

Checking values that violate business logic — negative payments, CSAT scores outside 1–5, dates outside the dataset window, or relationships that don't make sense (last_active before signup, end_date before start_date).

### 5.1 · MRR range — should be ≥ 0

In [22]:

q("""
SELECT
    MIN(mrr) AS min_mrr,
    MAX(mrr) AS max_mrr,
    SUM(CASE WHEN mrr < 0 THEN 1 ELSE 0 END) AS negative_mrr_rows
FROM subscriptions
""")

,min_mrr,max_mrr,negative_mrr_rows
0,0,499,0.0


### 5.2 · Payment amount range

In [23]:
q("""
SELECT
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    SUM(CASE WHEN amount < 0 THEN 1 ELSE 0 END) AS negative_amount_rows
FROM payments
""")

,min_amount,max_amount,negative_amount_rows
0,29,499,0.0


### 5.3 · CSAT scores — should be 1–5

In [24]:
q("""
SELECT
    csat_score,
    COUNT(*) AS occurrences
FROM support_tickets
WHERE csat_score IS NOT NULL
GROUP BY csat_score
ORDER BY csat_score
""")

,csat_score,occurrences
0,1.0,13
1,2.0,190
2,3.0,1202
3,4.0,2597
4,5.0,1700


### 5.4 · Date sanity — dates outside dataset window (Jun 2023 – May 2025)

In [25]:
q("""
SELECT
    'customers.signup_date < 2023-06-01' AS check_name,
    COUNT(*) AS violations
FROM customers WHERE signup_date < '2023-06-01'

UNION ALL SELECT 'customers.signup_date > 2025-05-31',
    COUNT(*) FROM customers WHERE signup_date > '2025-05-31'

UNION ALL SELECT 'subscriptions.start_date < 2023-06-01',
    COUNT(*) FROM subscriptions WHERE start_date < '2023-06-01'

UNION ALL SELECT 'subscriptions.start_date > 2025-05-31',
    COUNT(*) FROM subscriptions WHERE start_date > '2025-05-31'
""")

,check_name,violations
0,customers.signup_date < 2023-06-01,0
1,customers.signup_date > 2025-05-31,0
2,subscriptions.start_date < 2023-06-01,0
3,subscriptions.start_date > 2025-05-31,0


### 5.5 · Logical date violations

In [26]:
# Users whose last_active_date is before their signup_date
q("""
SELECT COUNT(*) AS impossible_user_dates
FROM users
WHERE last_active_date < signup_date
""")

,impossible_user_dates
0,0


In [27]:
# Subscriptions where end_date < start_date — impossible
q("""
SELECT COUNT(*) AS impossible_sub_dates
FROM subscriptions
WHERE end_date IS NOT NULL AND end_date < start_date
""")

,impossible_sub_dates
0,0


In [28]:
# Payments paid before they were invoiced — impossible
q("""
SELECT COUNT(*) AS impossible_payment_dates
FROM payments
WHERE paid_date IS NOT NULL AND paid_date < invoice_date
""")

,impossible_payment_dates
0,0


**Decision:**

> *no violations on any of these.*

---
## 6 · Format & Consistency

Check text fields for inconsistencies that would break grouping or matching. Classic example: "USA" vs "United States" vs "U.S.A." all referring to the same country.

### 6.1 · Email quality — whitespace and casing

In [29]:
# Emails with leading/trailing whitespace
q("""
SELECT email, LENGTH(email) - LENGTH(TRIM(email)) AS whitespace_chars
FROM users
WHERE email != TRIM(email)
LIMIT 10
""")

,email,whitespace_chars
0,michael.cooper@boyd.info,4
1,scott.mcknight@clark.org,4
2,briana.harrington@perry.org,4
3,barbara.moore@anderson.biz,4
4,kenneth.eaton@wong.net,4
5,heidi.patterson@clarke-james.org,4
6,zachary.holder@smith-joyce.com,4
7,joseph.jordan@glass.biz,4
8,thomas.alvarez@hall.com,4
9,christine.huffman@romero.com,4


In [30]:
# Emails not in standard lowercase
q("""
SELECT email
FROM users
WHERE email != LOWER(email)
LIMIT 10
""")

,email


In [31]:
# How many emails have either issue?
q("""
SELECT
    SUM(CASE WHEN email != TRIM(email) THEN 1 ELSE 0 END) AS has_whitespace,
    SUM(CASE WHEN email != LOWER(email) THEN 1 ELSE 0 END) AS has_mixed_case,
    COUNT(*) AS total_users
FROM users
""")

,has_whitespace,has_mixed_case,total_users
0,189.0,0.0,9314


### 6.2 · Country naming consistency

In [32]:
q("""
SELECT country, COUNT(*) AS rows_num
FROM customers
GROUP BY country
ORDER BY country
""")

,country,rows_num
0,Australia,41
1,Brazil,32
2,Canada,56
3,Egypt,14
4,France,43
5,Germany,68
6,India,26
7,Japan,23
8,Mexico,25
9,Netherlands,35


**Look for:** multiple spellings of the same country (e.g., 'USA' and 'United States'), inconsistent capitalization, leading/trailing spaces.

### 6.3 · Plan name consistency

In [33]:
q("""
SELECT plan_name, COUNT(*) AS rows_num
FROM subscriptions
GROUP BY plan_name
ORDER BY plan_name
""")

,plan_name,rows_num
0,Business,109
1,Enterprise,53
2,Free Trial,800
3,Pro,97
4,Starter,46


### 6.4 · Status values across tables

In [34]:
q("""
SELECT 'subscriptions' AS source, status, COUNT(*) AS rows_num FROM subscriptions GROUP BY status
UNION ALL
SELECT 'support_tickets', status, COUNT(*) FROM support_tickets GROUP BY status
UNION ALL
SELECT 'payments', status, COUNT(*) FROM payments GROUP BY status
ORDER BY source, status
""")

,source,status,rows_num
0,payments,failed,22
1,payments,paid,2548
2,subscriptions,active,180
3,subscriptions,churned,66
4,subscriptions,downgraded,23
5,subscriptions,ended,800
6,subscriptions,upgraded,36
7,support_tickets,open,20
8,support_tickets,resolved,8745


---
## 7 · Logical / Business Consistency

These checks verify the data tells a coherent business story. Different from the value-range checks above — these involve relationships *between* records.

### 7.1 · Each customer should have at most one currently-active subscription

In [35]:
q("""
SELECT customer_id, COUNT(*) AS active_subs
FROM subscriptions
WHERE end_date IS NULL
GROUP BY customer_id
HAVING COUNT(*) > 1
""")

,customer_id,active_subs


### 7.2 · MRR values should match the plan's expected price

In [36]:
q("""
SELECT plan_name, mrr, COUNT(*) AS rows_num
FROM subscriptions
GROUP BY plan_name, mrr
ORDER BY plan_name, mrr
""")

,plan_name,mrr,rows_num
0,Business,199,109
1,Enterprise,499,53
2,Free Trial,0,800
3,Pro,79,97
4,Starter,29,46


### 7.3 · Free Trial subscriptions should be ~14 days

In [37]:
q("""
SELECT
    DATEDIFF(end_date, start_date) AS trial_length_days,
    COUNT(*) AS rows_num
FROM subscriptions
WHERE plan_name = 'Free Trial' AND end_date IS NOT NULL
GROUP BY trial_length_days
ORDER BY trial_length_days
""")

,trial_length_days,rows_num
0,14,800


### 7.4 · Resolved tickets must have a resolved_date

In [38]:
q("""
SELECT COUNT(*) AS resolved_without_date
FROM support_tickets
WHERE status = 'resolved' AND resolved_date IS NULL
""")

,resolved_without_date
0,0


### 7.5 · Failed payments should have NULL paid_date

In [39]:
q("""
SELECT
    status,
    SUM(CASE WHEN paid_date IS NULL THEN 1 ELSE 0 END) AS null_paid_date,
    SUM(CASE WHEN paid_date IS NOT NULL THEN 1 ELSE 0 END) AS has_paid_date,
    COUNT(*) AS total
FROM payments
GROUP BY status
""")

,status,null_paid_date,has_paid_date,total
0,paid,0.0,2548.0,2548
1,failed,22.0,0.0,22
